# Plate Detection Dataset

Merges the full-frame plate annotations from `bangla-licence-plates` into one single-class YOLO dataset (`license_plate`), removes training images that duplicate validation or test images, and publishes the result as `bangla-plate-detection`.

| Source | Format | Plate class used |
|---|---|---|
| `bd-lpr/License Plate Localization` | Pascal VOC | `0` |
| `bd-lpr-blpd/LPD_Dataset` | YOLO | `0` (`License_Plate`) |
| `roboflow-dkqe3` | YOLO | `2` (`numberplate`); vehicle classes are dropped |

**Setup:** add the `bangla-licence-plates` dataset as an input.

In [ ]:
%pip install -q imagehash

## Setup

In [ ]:
import json
import random
import shutil
import xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

import imagehash
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

SOURCE = next(p for p in Path("/kaggle/input").rglob("bangla-licence-plates") if p.is_dir())
OUT = Path("/kaggle/working/plate-detection")
SPLITS = ("train", "val", "test")
IMAGE_EXT = {".jpg", ".jpeg", ".png"}
HAMMING_THRESHOLD = 4

if OUT.exists():
    shutil.rmtree(OUT)
for split in SPLITS:
    (OUT / "images" / split).mkdir(parents=True)
    (OUT / "labels" / split).mkdir(parents=True)

print("Source:", SOURCE)

## Read annotations

Each sample keeps its source split and its plate boxes in YOLO format (`x_center y_center width height`, normalised). Polygon labels are converted to their bounding boxes. Images without plates are kept as negatives.

In [ ]:
def read_voc(image_path: Path, plate_names: set):
    root = ET.parse(image_path.with_suffix(".xml")).getroot()
    size = root.find("size")
    if size is not None and int(size.findtext("width", 0)) > 0:
        width, height = int(size.findtext("width")), int(size.findtext("height"))
    else:
        width, height = Image.open(image_path).size
    boxes = []
    for obj in root.iter("object"):
        if obj.findtext("name") not in plate_names:
            continue
        b = obj.find("bndbox")
        x1, y1, x2, y2 = (float(b.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
        x1, x2 = max(0.0, x1), min(float(width), x2)
        y1, y2 = max(0.0, y1), min(float(height), y2)
        if x2 > x1 and y2 > y1:
            boxes.append(((x1 + x2) / 2 / width, (y1 + y2) / 2 / height, (x2 - x1) / width, (y2 - y1) / height))
    return boxes


def read_yolo(image_path: Path, plate_ids: set):
    label = image_path.parent.parent / "labels" / f"{image_path.stem}.txt"
    boxes = []
    if label.exists():
        for line in label.read_text().splitlines():
            parts = line.split()
            if not parts or parts[0] not in plate_ids:
                continue
            values = [float(v) for v in parts[1:]]
            if len(values) == 4:
                boxes.append(tuple(values))
            elif len(values) >= 6 and len(values) % 2 == 0:
                xs, ys = values[0::2], values[1::2]
                x1, x2 = max(0.0, min(xs)), min(1.0, max(xs))
                y1, y2 = max(0.0, min(ys)), min(1.0, max(ys))
                if x2 > x1 and y2 > y1:
                    boxes.append(((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1))
    return boxes


SOURCES = [
    # name, root, {source split: target split}, reader, plate labels
    ("bd-lpr", SOURCE / "bd-lpr" / "License Plate Localization",
     {"train": "train", "validation": "val", "test": "test"}, read_voc, {"0"}),
    ("blpd", SOURCE / "bd-lpr-blpd" / "LPD_Dataset",
     {"train": "train", "valid": "val", "test": "test"}, read_yolo, {"0"}),
    ("dkqe3", SOURCE / "roboflow-dkqe3",
     {"train": "train", "valid": "val", "test": "test"}, read_yolo, {"2"}),
]

samples = []
for name, root, split_map, reader, plate_labels in SOURCES:
    for source_split, split in split_map.items():
        folder = root / source_split
        images = [p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXT]
        for image in tqdm(images, desc=f"{name}/{source_split}", leave=False):
            samples.append({"source": name, "split": split, "image": image, "boxes": reader(image, plate_labels)})

print(f"{len(samples)} images, {sum(len(s['boxes']) for s in samples)} plate boxes")

## Remove duplicates across splits

Images are compared by perceptual hash. A training image within `HAMMING_THRESHOLD` bits of any validation or test image is removed, and a validation image close to a test image is removed, so evaluation images never appear in training.

In [ ]:
def phash(path: Path) -> int:
    with Image.open(path) as image:
        return int(str(imagehash.phash(image.convert("L"))), 16)


for s in tqdm(samples, desc="hashing"):
    s["hash"] = phash(s["image"])


def near_duplicates(candidates: list, references: list) -> set:
    if not candidates or not references:
        return set()
    ref = np.array([r["hash"] for r in references], dtype=np.uint64)
    removed = set()
    for i, s in enumerate(candidates):
        xor = np.bitwise_xor(ref, np.uint64(s["hash"]))
        distance = np.unpackbits(xor.view(np.uint8).reshape(-1, 8), axis=1).sum(axis=1)
        if distance.min() <= HAMMING_THRESHOLD:
            removed.add(i)
    return removed


by_split = {split: [s for s in samples if s["split"] == split] for split in SPLITS}

drop_val = near_duplicates(by_split["val"], by_split["test"])
by_split["val"] = [s for i, s in enumerate(by_split["val"]) if i not in drop_val]

drop_train = near_duplicates(by_split["train"], by_split["val"] + by_split["test"])
by_split["train"] = [s for i, s in enumerate(by_split["train"]) if i not in drop_train]

print(f"Removed {len(drop_val)} validation and {len(drop_train)} training images")

## Write the dataset

In [ ]:
summary = Counter()
for split, items in by_split.items():
    for s in tqdm(items, desc=split):
        stem = f"{s['source']}__{s['image'].stem}"
        shutil.copy2(s["image"], OUT / "images" / split / f"{stem}{s['image'].suffix.lower()}")
        lines = [f"0 {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for x, y, w, h in s["boxes"]]
        (OUT / "labels" / split / f"{stem}.txt").write_text("\n".join(lines))
        summary[(split, s["source"], "images")] += 1
        summary[(split, s["source"], "boxes")] += len(s["boxes"])
        summary[(split, s["source"], "negatives")] += int(not s["boxes"])

(OUT / "data.yaml").write_text(
    "train: images/train\n"
    "val: images/val\n"
    "test: images/test\n"
    "names:\n"
    "  0: license_plate\n"
)

rows = []
print(f"{'Split':6} {'Source':8} {'Images':>7} {'Boxes':>7} {'Negatives':>10}")
for split in SPLITS:
    for name, *_ in SOURCES:
        images, boxes, negatives = (summary[(split, name, k)] for k in ("images", "boxes", "negatives"))
        rows.append({"split": split, "source": name, "images": images, "boxes": boxes, "negatives": negatives})
        print(f"{split:6} {name:8} {images:7d} {boxes:7d} {negatives:10d}")
    total = sum(summary[(split, n, 'images')] for n, *_ in SOURCES)
    print(f"{split:6} {'total':8} {total:7d}")

_ = (OUT / "summary.json").write_text(json.dumps({
    "rows": rows,
    "removed_duplicates": {"val": len(drop_val), "train": len(drop_train)},
    "hamming_threshold": HAMMING_THRESHOLD,
}, indent=2))

## Samples

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

random.seed(0)
fig, axes = plt.subplots(len(SPLITS), 5, figsize=(20, 4 * len(SPLITS)), squeeze=False)
for row, split in zip(axes, SPLITS):
    images = random.sample(sorted((OUT / "images" / split).iterdir()), 5)
    for ax, path in zip(row, images):
        image = Image.open(path).convert("RGB")
        width, height = image.size
        ax.imshow(image)
        for line in (OUT / "labels" / split / f"{path.stem}.txt").read_text().splitlines():
            _, x, y, w, h = map(float, line.split())
            ax.add_patch(patches.Rectangle(((x - w / 2) * width, (y - h / 2) * height), w * width, h * height,
                                           fill=False, color="lime", lw=2))
        ax.set_title(f"{split}: {path.name[:30]}", fontsize=8)
        ax.axis("off")
plt.tight_layout()
plt.show()